# Training Details of Models

We provide the configuration files used in training. Hyper-parameters' names are mostly self-explanatory. 

In [1]:
from math import ceil
import numpy as np
import torch
import torch.nn as nn

sampling_rate = 22050
frame_length = 128
mel_length = 80

## Learning Rate

In all experiments we used the following LR scheduling method:

In [2]:
def noam_lr(warmup_steps, min_lr, init_lr, step, power=0.35):
    return np.maximum(
        init_lr * warmup_steps ** power * np.minimum(
            step * warmup_steps ** (-1 - power), 
            step ** -power
        ), 
        min_lr
    )

## STFT Loss

We used following STFTLoss Definition in PyTorch.

In [3]:
class STFTLoss(nn.Module):
    def __init__(self, fft_lengths, window_lengths, hop_lengths, loss_scale_type):
        """
        STFT Loss
        fft_lengths: list of int
        window_lengths: list of int
        hop_lengths: list of int
        loss_scale_type: str defining the scale of loss
        """
        super(STFTLoss, self).__init__()
        self.fft_lengths = fft_lengths
        self.window_lengths = window_lengths
        self.hop_lengths = hop_lengths
        self.loss_scale_type = loss_scale_type
        
    def forward(self, x, y):
        """
        x: FloatTensor [Batch, 1, T]
        y: FloatTensor [Batch, 1, T]
        returns: FloatTensor [] as total loss
        """
        x, y = x.squeeze(1), y.squeeze(1)
        loss = 0.0
        batch_size = x.size(0)
        z = torch.cat([x, y], dim=0) # [2 x Batch, T]
        for fft_length, window_length, hop_length in zip(self.fft_lengths, \
            self.window_lengths, self.hop_lengths):
            window = torch.hann_window(window_length)
            Z = torch.stft(z, fft_length, hop_length, window_length, window) # [2 x Batch, Frame, 2]
            SquareZ = Z.pow(2).sum(dim=-1) + 1e-10 # [2 x Batch, Frame]
            SquareX, SquareY = SquareZ.split(batch_size, dim=0)
            MagZ = SquareZ.sqrt() # NOTICE: read more about STFT Loss
            MagX, MagY = MagZ.split(batch_size, dim=0)
            if self.loss_scale_type == "log_linear":
                loss += (MagX - MagY).abs().mean() + 0.5 * (SquareX.log() - SquareY.log()).abs().mean()
            elif self.loss_scale_type == "linear":
                loss += (MagX - MagY).abs().mean()
            else:
                raise RuntimeError(f"Unrecognized loss scale type {self.loss_scale_type}")
        return loss


## NHV

The NHV model is trained 600K steps.

In [4]:
class NHVConfig:
    #### Batch Loading Configs
    batch_size = 1
    segment_length_s = 4
    segment_length_frames = ceil(segment_length_s * sampling_rate / frame_length)
    segment_length = segment_length_frames * frame_length

    #### Maximum Number of Harmonics
    source_harmonics = 100

    #### Training
    # Generator
    g_warmup_steps = 4000
    g_decay_lr = True
    g_init_lr = 0.0006
    g_min_lr = 0.00001
    g_grad_clip = True
    g_grad_clip_max_norm = 100.0
    # Discriminator
    d_warmup_steps = 20000
    d_decay_lr = True
    d_init_lr = 0.0002
    d_min_lr = 0.00001
    # Adversarial Loss Ratio, scales up L_G and L_D by adv_ratio 
    adv_ratio = 4.0

    #### STFT Loss configuration
    loss_window_lengths = list(range(128, 1024, 128)) + list(range(1024, 2048, 512)) + list(range(2048, 5000, 1024))
    loss_fft_lengths = [wnd * 2 for wnd in loss_window_lengths]
    loss_hop_lengths = [wnd // 4 for wnd in loss_window_lengths]
    loss_scale_type = "log_linear"

    #### FFT size used in converting cepstrums
    cepstrum_fft_length = 1024 # The impulse Response would be -1024 ~ + 1024 of length 2049

    #### Length of Cepstrums
    # Harmonic:
    harmonic_time_ms = 5
    harmonic_max_quefrency = ceil(sampling_rate / 1000 * harmonic_time_ms)
    # Noise:
    noise_time_ms = 5
    noise_max_quefrency = ceil(sampling_rate / 1000 * noise_time_ms)

    #### Network Hidden Size
    hidden_size = 256

    #### Reverb with exponential decay, x'[n] = 0.995 ^ n * x[n]
    reverb_length = 1024
    reverb_decay = 0.995

    #### Discriminator Structure
    d_residual_channels = 64
    d_gate_channels = 128
    d_skip_channels = 64
    d_kernel_size = 3
    d_cond_channels = mel_length
    d_layers = 14
    d_stacks = 2
    d_is_causal = False
    n_discriminators = 1

    #### Whether to use Pitch Adaptive Liftering
    liftering = False

## b-NSF-adv
The b-NSF-adv model is trained 350K steps.

In [5]:
class NSFConfig:
    batch_size = 1
    segment_length_s = 3
    segment_length_frames = ceil(segment_length_s * sampling_rate / frame_length)
    segment_length = segment_length_frames * frame_length

    #### Training
    # Generator
    g_warmup_steps = 4000
    g_decay_lr = True
    g_init_lr = 0.001
    g_min_lr = 0.00002
    g_grad_clip = False
    g_grad_clip_max_norm = 20.0
    # Discriminator
    d_warmup_steps = 20000
    d_decay_lr = True
    d_init_lr = 0.0002
    d_min_lr = 0.00001
    # Adversarial Loss Ratio, scales up L_G and L_D by adv_ratio 
    adv_ratio = 0.1

    #### Architecture
    layers = [10, 10, 10, 10, 10]
    feature_channels = mel_length
    condition_channels = 64
    residual_channels = 64
    kernel_size = 3
    upsample_kernel_size = 3

    #### STFT Loss
    loss_window_lengths = [16, 32, 64, 128, 256, 512, 1024, 2048]
    loss_fft_lengths = [wnd * 2 for wnd in loss_window_lengths]
    loss_hop_lengths = [wnd // 4 for wnd in loss_window_lengths]
    loss_scale_type = "linear"

    #### Discriminator Structure
    discriminator_configs = [
        [  # inc,  outc, kernel, dilation, stride, padding
            (1,    64,   3,      1,        2,      0),
            (64,   64,   3,      1,        2,      0),
            (64,   64,   3,      1,        4,      0),
            (64,   64,   3,      1,        2,      0),
            (64,   64,   3,      1,        2,      0),
            (64,   64,   3,      1,        2,      0),
            (64,   64,   3,      1,        1,      0),
            (64,   64,   3,      1,        1,      0),
            (64,   64,   3,      1,        1,      0),
            (64,    1,   3,      1,        1,      0)
        ]
    ]

    n_discriminators = len(discriminator_configs)


## Parallel WaveGAN
The Parallel WaveGAN model is trained 400K steps.

In [6]:
class PWGConfig:
    batch_size = 1
    segment_length_s = 3
    segment_length_frames = ceil(segment_length_s * sampling_rate / frame_length)
    segment_length = segment_length_frames * frame_length

    #### Source Model
    source_harmonics = 100

    #### Training
    # Generator
    g_warmup_steps = 4000
    g_decay_lr = True
    g_init_lr = 0.001
    g_min_lr = 0.00002
    g_grad_clip = True
    g_grad_clip_max_norm = 20.0
    # Discriminator
    d_warmup_steps = 20000
    d_decay_lr = True
    d_init_lr = 0.0005
    d_min_lr = 0.00001
    # Adversarial Loss Ratio, scales up L_G and L_D by adv_ratio 
    adv_ratio = 0.4

    #### WaveNet Architecture
    condition_channels = mel_length
    layers = 30
    stacks = 3
    residual_channels = 64
    gate_channels = 128
    skip_channels = 64
    kernel_size = 3
    upsample_factor = [8, 16]

    #### STFT Loss
    loss_window_lengths = [16, 32, 64, 128, 256, 512, 1024, 2048]
    loss_fft_lengths = [wnd * 2 for wnd in loss_window_lengths]
    loss_hop_lengths = [wnd // 4 for wnd in loss_window_lengths]
    loss_scale_type = "linear"

    #### Discriminator Structure
    discriminator_configs = [
        [  # inc,  outc, kernel, dilation, stride, padding
            (1,    64,   3,      1,        1,      0),
            (64,   64,   3,      1,        1,      0),
            (64,   64,   3,      2,        1,      0),
            (64,   64,   3,      3,        1,      0),
            (64,   64,   3,      4,        1,      0),
            (64,   64,   3,      5,        1,      0),
            (64,   64,   3,      6,        1,      0),
            (64,   64,   3,      7,        1,      0),
            (64,   64,   3,      8,        1,      0),
            (64,    1,   3,      1,        1,      0)
        ]
    ]

    n_discriminators = len(discriminator_configs)
